# Figure 2 — Passive electrical consequences of OPC morphology

This notebook merges the **GM4 cohort-wide passive simulations** with the morphology table generated for Figure 1. It reproduces all analyses, selects representative cells objectively, performs inferential statistics, and exports figure source data.

**Cohort:** 35 experimentally reconstructed OPCs (P10: 12; P20: 12; P50: 11).

**Baseline passive parameters:** $R_m=20{,}000\,\Omega\cdot\mathrm{cm}^2$, $R_a=150\,\Omega\cdot\mathrm{cm}$, $C_m=1\,\mu\mathrm{F}/\mathrm{cm}^2$, $E_{pas}=-75$ mV, uniform process diameter $0.3\,\mu$m.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
GM4 = Path(r"/mnt/data/fig2_work/4-opc_step4_cohort_passive/opc_step4_passive_results")
MORPH_PATH = Path(r"/mnt/data/figure1_outputs/morphology_master_table.csv")
OUT = Path(r"/mnt/data/figure2_outputs")
OUT.mkdir(exist_ok=True)
AGE_ORDER = ['P10','P20','P50']
plt.rcParams.update({'font.size':9,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':120})


## 1. Load and validate the cohort

In [ ]:
passive = pd.read_csv(GM4/'population'/'cell_level_passive_metrics.csv')
morph = pd.read_csv(MORPH_PATH).rename(columns={'age':'age_group'})
merged = passive.merge(morph, on=['cell_id','age_group'], how='left', validate='one_to_one')
print('Passive rows:', len(passive))
print('Merged rows:', len(merged))
print(merged.groupby('age_group').size().reindex(AGE_ORDER))
assert len(merged)==35
assert merged['total_process_length_um'].notna().all()
assert merged['resting_stable'].all()
assert (merged['tau_fit_r2']>0.99).all()
merged.to_csv(OUT/'Figure2_integrated_source_data.csv', index=False)
merged.head()

## 2. Descriptive and inferential statistics

Age-group comparisons use Kruskal–Wallis tests followed by pairwise Mann–Whitney tests with Holm correction. Spearman correlations quantify morphology–electrical relationships. Effect sizes for pairwise comparisons are rank-biserial correlations.

In [ ]:
metrics = {
 'input_resistance_MOhm':'Input resistance (MΩ)',
 'tau_ms':'Membrane time constant (ms)',
 'minimum_soma_normalised_attenuation':'Minimum soma-normalised attenuation',
 'mean_soma_normalised_attenuation':'Mean soma-normalised attenuation'
}
rows=[]; pair_rows=[]
for col,label in metrics.items():
    groups=[merged.loc[merged.age_group==a,col].dropna() for a in AGE_ORDER]
    H,p=stats.kruskal(*groups)
    rows.append({'metric':col,'label':label,'test':'Kruskal-Wallis','statistic':H,'p_value':p})
    raw=[]
    for i,a in enumerate(AGE_ORDER):
        for b in AGE_ORDER[i+1:]:
            x=merged.loc[merged.age_group==a,col].dropna(); y=merged.loc[merged.age_group==b,col].dropna()
            U,pairp=stats.mannwhitneyu(x,y,alternative='two-sided')
            rbc=1-2*U/(len(x)*len(y))
            raw.append([a,b,U,pairp,rbc])
    adj=multipletests([r[3] for r in raw],method='holm')[1]
    for r,pa in zip(raw,adj):
        pair_rows.append({'metric':col,'group_1':r[0],'group_2':r[1],'U':r[2],'p_raw':r[3],'p_holm':pa,'rank_biserial':r[4]})
overall=pd.DataFrame(rows); pairwise=pd.DataFrame(pair_rows)
desc=merged.groupby('age_group')[list(metrics)].agg(['count','mean','std','median','min','max']).reindex(AGE_ORDER)
overall.to_csv(OUT/'Figure2_overall_tests.csv',index=False)
pairwise.to_csv(OUT/'Figure2_pairwise_tests.csv',index=False)
desc.to_csv(OUT/'Figure2_descriptive_statistics.csv')
display(overall)
display(pairwise.sort_values(['metric','p_holm']))

## 3. Morphology–electrical relationships

In [ ]:
predictors=['total_process_length_um','max_node_path_distance_um','n_terminal_segments','n_branch_segments']
corr=[]
for x in predictors:
    rho,p=stats.spearmanr(merged[x],merged['input_resistance_MOhm'],nan_policy='omit')
    corr.append({'outcome':'input_resistance_MOhm','predictor':x,'spearman_rho':rho,'p_value':p})
corr=pd.DataFrame(corr)
corr['p_holm']=multipletests(corr.p_value,method='holm')[1]
corr.to_csv(OUT/'Figure2_morphology_correlations.csv',index=False)
display(corr.sort_values('p_holm'))

# Exploratory standardised multivariable OLS; descriptors chosen a priori and kept few for n=35.
Xcols=['total_process_length_um','max_node_path_distance_um','n_terminal_segments']
X=pd.DataFrame(StandardScaler().fit_transform(merged[Xcols]),columns=Xcols,index=merged.index)
X=sm.add_constant(X)
model=sm.OLS(merged['input_resistance_MOhm'],X).fit(cov_type='HC3')
coef=pd.DataFrame({'term':model.params.index,'coefficient':model.params.values,'SE_HC3':model.bse.values,'p_value':model.pvalues.values})
coef.to_csv(OUT/'Figure2_exploratory_regression.csv',index=False)
print(model.summary())

## 4. Representative cells and voltage traces

For visual continuity with Figure 1, the representative cells are those selected there as closest to the multivariate age-group median: NX64_2_P10, NX16_1_P20, and NX90_20_P50.

In [ ]:
representatives={'P10':'NX64_2_P10','P20':'NX16_1_P20','P50':'NX90_20_P50'}
trace_frames=[]
for age,cid in representatives.items():
    f=GM4/'cells'/cid/f'{cid}_soma_trace.csv'
    tr=pd.read_csv(f)
    tr['cell_id']=cid; tr['age_group']=age
    tr['delta_V_mV']=tr['soma_mV']-tr['soma_mV'].iloc[0]
    trace_frames.append(tr)
traces=pd.concat(trace_frames,ignore_index=True)
traces.to_csv(OUT/'Figure2_representative_traces.csv',index=False)
merged.loc[merged.cell_id.isin(representatives.values()),['cell_id','age_group','input_resistance_MOhm','tau_ms','minimum_soma_normalised_attenuation']]

## 5. Draft Figure 2

In [ ]:
def grouped_points(ax,col,ylabel):
    vals=[merged.loc[merged.age_group==a,col].dropna().values for a in AGE_ORDER]
    parts=ax.violinplot(vals,positions=np.arange(3),showmeans=False,showmedians=False,showextrema=False)
    for pc in parts['bodies']: pc.set_alpha(0.25)
    rng=np.random.default_rng(20260727)
    for i,v in enumerate(vals):
        ax.scatter(np.full(len(v),i)+rng.uniform(-.09,.09,len(v)),v,s=20,zorder=3)
        ax.plot([i-.18,i+.18],[np.median(v)]*2,lw=2)
    ax.set_xticks(range(3),AGE_ORDER); ax.set_ylabel(ylabel)

def scatter_fit(ax,x,y,xlabel,ylabel):
    for age,marker in zip(AGE_ORDER,['o','s','^']):
        d=merged[merged.age_group==age]
        ax.scatter(d[x],d[y],label=age,marker=marker,s=32)
    ok=merged[[x,y]].dropna(); slope,intercept,r,p,se=stats.linregress(ok[x],ok[y])
    xx=np.linspace(ok[x].min(),ok[x].max(),100)
    ax.plot(xx,intercept+slope*xx,lw=1.5)
    rho,ps=stats.spearmanr(ok[x],ok[y])
    ax.text(.04,.96,f'Spearman ρ = {rho:.2f}\np = {ps:.3g}',transform=ax.transAxes,va='top')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)

fig,axs=plt.subplots(2,3,figsize=(11.2,6.8),constrained_layout=True)
# A
ax=axs[0,0]
for age in AGE_ORDER:
    d=traces[traces.age_group==age]
    ax.plot(d.time_ms,d.delta_V_mV,label=f"{age}: {representatives[age]}")
ax.axhline(0,lw=.6); ax.set_xlabel('Time (ms)'); ax.set_ylabel('Somatic ΔV (mV)'); ax.legend(frameon=False,fontsize=7)
# B-D
grouped_points(axs[0,1],'input_resistance_MOhm','Input resistance (MΩ)')
grouped_points(axs[0,2],'tau_ms','Membrane time constant (ms)')
grouped_points(axs[1,0],'minimum_soma_normalised_attenuation','Minimum soma-normalised attenuation')
# E-F
scatter_fit(axs[1,1],'total_process_length_um','input_resistance_MOhm','Total process length (µm)','Input resistance (MΩ)')
scatter_fit(axs[1,2],'max_node_path_distance_um','input_resistance_MOhm','Maximum path distance (µm)','Input resistance (MΩ)')
for label,ax in zip('ABCDEF',axs.flat): ax.text(-.16,1.08,label,transform=ax.transAxes,fontweight='bold',fontsize=12,va='top')
fig.suptitle('Passive electrical consequences of developmental OPC morphology',fontsize=13)
for ext in ['pdf','png','svg']:
    fig.savefig(OUT/f'Figure2_passive_electrical_properties.{ext}',dpi=400 if ext=='png' else None,bbox_inches='tight')
plt.show()

## 6. Analysis summary

In [ ]:
print('Cohort:', merged.groupby('age_group').size().reindex(AGE_ORDER).to_dict())
print()
print('Overall tests:')
display(overall)
print()
print('Representative cells:', representatives)
print()
print('All source data and statistics exported to:', OUT)
